In [1]:
import csv
import os
from datetime import datetime


PRODUCT_FILE = "products.csv"
BILL_FILE = "bill.txt"


# ============================================================
# Product Class
# ============================================================

class Product:
    def __init__(self, product_id, name, category, price, stock):
        self.product_id = product_id
        self.name = name
        self.category = category
        self.price = price
        self.stock = stock

    def display(self):
        print(
            f"{self.product_id:<8}"
            f"{self.name:<20}"
            f"{self.category:<18}"
            f"₹{self.price:<12.2f}"
            f"{self.stock:<8}"
        )


# ============================================================
# Shopping Cart Class
# ============================================================

class ShoppingCart:
    def __init__(self):
        # Dictionary:
        # product_id -> quantity
        self.items = {}

    def add_item(self, product, quantity):
        if quantity <= 0:
            print("Quantity must be greater than 0.")
            return

        current_quantity = self.items.get(product.product_id, 0)

        if current_quantity + quantity > product.stock:
            print(
                f"Only {product.stock - current_quantity} "
                f"unit(s) available."
            )
            return

        self.items[product.product_id] = current_quantity + quantity

        print(
            f"{quantity} x {product.name} "
            f"added to cart successfully."
        )

    def remove_item(self, product_id):
        if product_id in self.items:
            del self.items[product_id]
            print("Product removed from cart.")
        else:
            print("Product is not in the cart.")

    def update_quantity(self, product, quantity):
        if product.product_id not in self.items:
            print("Product is not in the cart.")
            return

        if quantity <= 0:
            print("Quantity must be greater than 0.")
            return

        if quantity > product.stock:
            print(f"Only {product.stock} unit(s) available.")
            return

        self.items[product.product_id] = quantity
        print("Cart quantity updated successfully.")

    def is_empty(self):
        return len(self.items) == 0

    def calculate_total(self, products):
        total = 0

        for product_id, quantity in self.items.items():
            product = products.get(product_id)

            if product:
                total += product.price * quantity

        return total

    def display_cart(self, products):
        if self.is_empty():
            print("\nYour cart is empty.")
            return

        print("\n" + "=" * 75)
        print("                         YOUR CART")
        print("=" * 75)

        print(
            f"{'ID':<8}"
            f"{'Product':<20}"
            f"{'Price':<12}"
            f"{'Qty':<8}"
            f"{'Subtotal':<15}"
        )

        print("-" * 75)

        for product_id, quantity in self.items.items():

            product = products.get(product_id)

            if product:
                subtotal = product.price * quantity

                print(
                    f"{product.product_id:<8}"
                    f"{product.name:<20}"
                    f"₹{product.price:<11.2f}"
                    f"{quantity:<8}"
                    f"₹{subtotal:<14.2f}"
                )

        print("-" * 75)

        total = self.calculate_total(products)

        print(f"{'Total:':>60} ₹{total:.2f}")
        print("=" * 75)


# ============================================================
# Shopping System Class
# ============================================================

class ShoppingSystem:
    def __init__(self):
        self.products = {}
        self.cart = ShoppingCart()

        self.load_products()

    # --------------------------------------------------------
    # Load products from CSV
    # --------------------------------------------------------

    def load_products(self):

        if not os.path.exists(PRODUCT_FILE):
            self.create_sample_products()

        try:
            with open(PRODUCT_FILE, "r", newline="") as file:

                reader = csv.DictReader(file)

                for row in reader:

                    product = Product(
                        int(row["id"]),
                        row["name"],
                        row["category"],
                        float(row["price"]),
                        int(row["stock"])
                    )

                    self.products[product.product_id] = product

        except (ValueError, KeyError):
            print("Error: Invalid product data in CSV file.")

        except IOError:
            print("Error: Unable to read product file.")

    # --------------------------------------------------------
    # Create sample product file
    # --------------------------------------------------------

    def create_sample_products(self):

        products = [
            [101, "Laptop", "Electronics", 55000, 10],
            [102, "Mouse", "Electronics", 800, 25],
            [103, "Keyboard", "Electronics", 1500, 15],
            [104, "Headphones", "Electronics", 2000, 20],
            [105, "T-Shirt", "Clothing", 700, 30],
            [106, "Jeans", "Clothing", 1800, 15],
            [107, "Backpack", "Accessories", 1200, 20],
            [108, "Watch", "Accessories", 2500, 12]
        ]

        try:

            with open(
                PRODUCT_FILE,
                "w",
                newline=""
            ) as file:

                writer = csv.writer(file)

                writer.writerow(
                    ["id", "name", "category", "price", "stock"]
                )

                writer.writerows(products)

        except IOError:
            print("Error: Unable to create product file.")

    # --------------------------------------------------------
    # Display all products
    # --------------------------------------------------------

    def display_products(self):

        print("\n" + "=" * 75)
        print("                         PRODUCTS")
        print("=" * 75)

        print(
            f"{'ID':<8}"
            f"{'Name':<20}"
            f"{'Category':<18}"
            f"{'Price':<12}"
            f"{'Stock':<8}"
        )

        print("-" * 75)

        for product in self.products.values():
            product.display()

        print("=" * 75)

    # --------------------------------------------------------
    # Search product
    # --------------------------------------------------------

    def search_product(self):

        search_term = input(
            "\nEnter product name or category: "
        ).strip().lower()

        if not search_term:
            print("Search value cannot be empty.")
            return

        found_products = []

        for product in self.products.values():

            if (
                search_term in product.name.lower()
                or search_term in product.category.lower()
            ):
                found_products.append(product)

        if not found_products:
            print("No products found.")
            return

        print("\nSearch Results:")

        print(
            f"{'ID':<8}"
            f"{'Name':<20}"
            f"{'Category':<18}"
            f"{'Price':<12}"
            f"{'Stock':<8}"
        )

        print("-" * 75)

        for product in found_products:
            product.display()

    # --------------------------------------------------------
    # Add product to cart
    # --------------------------------------------------------

    def add_to_cart(self):

        try:

            product_id = int(
                input("Enter Product ID: ")
            )

            if product_id not in self.products:
                print("Product not found.")
                return

            product = self.products[product_id]

            quantity = int(
                input("Enter quantity: ")
            )

            self.cart.add_item(
                product,
                quantity
            )

        except ValueError:
            print("Please enter valid numeric values.")

    # --------------------------------------------------------
    # Remove product from cart
    # --------------------------------------------------------

    def remove_from_cart(self):

        try:

            product_id = int(
                input("Enter Product ID to remove: ")
            )

            self.cart.remove_item(product_id)

        except ValueError:
            print("Please enter a valid Product ID.")

    # --------------------------------------------------------
    # Update cart quantity
    # --------------------------------------------------------

    def update_cart(self):

        try:

            product_id = int(
                input("Enter Product ID: ")
            )

            if product_id not in self.products:
                print("Product not found.")
                return

            quantity = int(
                input("Enter new quantity: ")
            )

            product = self.products[product_id]

            self.cart.update_quantity(
                product,
                quantity
            )

        except ValueError:
            print("Please enter valid numeric values.")

    # --------------------------------------------------------
    # Apply discount
    # --------------------------------------------------------

    def calculate_discount(self, total):

        if total >= 10000:
            return total * 0.10

        elif total >= 5000:
            return total * 0.05

        return 0

    # --------------------------------------------------------
    # Checkout
    # --------------------------------------------------------

    def checkout(self):

        if self.cart.is_empty():
            print("\nYour cart is empty.")
            return

        total = self.cart.calculate_total(
            self.products
        )

        discount = self.calculate_discount(total)

        final_amount = total - discount

        print("\n" + "=" * 55)
        print("                    CHECKOUT")
        print("=" * 55)

        print(f"Subtotal       : ₹{total:.2f}")
        print(f"Discount       : ₹{discount:.2f}")
        print(f"Final Amount   : ₹{final_amount:.2f}")

        print("=" * 55)

        confirm = input(
            "Confirm purchase? (y/n): "
        ).strip().lower()

        if confirm != "y":
            print("Checkout cancelled.")
            return

        # Update stock
        for product_id, quantity in self.cart.items.items():

            product = self.products.get(product_id)

            if product:
                product.stock -= quantity

        # Save updated products
        self.save_products()

        # Generate bill
        self.generate_bill(
            total,
            discount,
            final_amount
        )

        self.cart.items.clear()

        print("\nPurchase successful!")
        print("Bill saved to bill.txt")

    # --------------------------------------------------------
    # Save updated product stock
    # --------------------------------------------------------

    def save_products(self):

        try:

            with open(
                PRODUCT_FILE,
                "w",
                newline=""
            ) as file:

                writer = csv.writer(file)

                writer.writerow(
                    ["id", "name", "category", "price", "stock"]
                )

                for product in self.products.values():

                    writer.writerow([
                        product.product_id,
                        product.name,
                        product.category,
                        product.price,
                        product.stock
                    ])

        except IOError:
            print("Error: Unable to save product data.")

    # --------------------------------------------------------
    # Generate bill
    # --------------------------------------------------------

    def generate_bill(
        self,
        total,
        discount,
        final_amount
    ):

        try:

            with open(
                BILL_FILE,
                "w"
            ) as file:

                file.write("=" * 55 + "\n")
                file.write("              SHOPPING CART BILL\n")
                file.write("=" * 55 + "\n")

                file.write(
                    f"Date: "
                    f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
                )

                file.write("-" * 55 + "\n")

                for product_id, quantity in self.cart.items.items():

                    product = self.products.get(product_id)

                    if product:

                        subtotal = (
                            product.price * quantity
                        )

                        file.write(
                            f"{product.name:<20}"
                            f"Qty: {quantity:<5}"
                            f"₹{subtotal:.2f}\n"
                        )

                file.write("-" * 55 + "\n")

                file.write(
                    f"Subtotal:       ₹{total:.2f}\n"
                )

                file.write(
                    f"Discount:       ₹{discount:.2f}\n"
                )

                file.write(
                    f"Final Amount:   ₹{final_amount:.2f}\n"
                )

                file.write("=" * 55 + "\n")

        except IOError:
            print("Error: Unable to generate bill.")

    # --------------------------------------------------------
    # Run application
    # --------------------------------------------------------

    def run(self):

        while True:

            print("\n")
            print("=" * 50)
            print("          🛒 SHOPPING CART SYSTEM")
            print("=" * 50)

            print("1. View Products")
            print("2. Search Product")
            print("3. Add Product to Cart")
            print("4. View Cart")
            print("5. Remove Product from Cart")
            print("6. Update Cart Quantity")
            print("7. Checkout")
            print("8. Exit")

            print("=" * 50)

            choice = input(
                "Enter your choice: "
            ).strip()

            if choice == "1":

                self.display_products()

            elif choice == "2":

                self.search_product()

            elif choice == "3":

                self.add_to_cart()

            elif choice == "4":

                self.cart.display_cart(
                    self.products
                )

            elif choice == "5":

                self.remove_from_cart()

            elif choice == "6":

                self.update_cart()

            elif choice == "7":

                self.checkout()

            elif choice == "8":

                print(
                    "\nThank you for using "
                    "Shopping Cart System!"
                )

                break

            else:

                print(
                    "Invalid choice. "
                    "Please select between 1 and 8."
                )


# ============================================================
# Program Entry Point
# ============================================================

if __name__ == "__main__":

    shopping_system = ShoppingSystem()

    shopping_system.run()



          🛒 SHOPPING CART SYSTEM
1. View Products
2. Search Product
3. Add Product to Cart
4. View Cart
5. Remove Product from Cart
6. Update Cart Quantity
7. Checkout
8. Exit
Enter your choice: 1

                         PRODUCTS
ID      Name                Category          Price       Stock   
---------------------------------------------------------------------------
101     Laptop              Electronics       ₹55000.00    10      
102     Mouse               Electronics       ₹800.00      25      
103     Keyboard            Electronics       ₹1500.00     15      
104     Headphones          Electronics       ₹2000.00     20      
105     T-Shirt             Clothing          ₹700.00      30      
106     Jeans               Clothing          ₹1800.00     15      
107     Backpack            Accessories       ₹1200.00     20      
108     Watch               Accessories       ₹2500.00     12      


          🛒 SHOPPING CART SYSTEM
1. View Products
2. Search Product
3. Add Pr